# Project: Multi-Modal Airline AI Assistant

This notebook takes the FlightAI airline assistant from tool-calling into full
**multi-modal territory**: it can generate images, speak its replies out loud, and
run inside a fully custom Gradio UI instead of the default chat widget.

**What you'll learn:**
1. A recap of the tool-calling assistant (database-backed ticket prices).
2. What Gradio actually does under the hood, end to end.
3. **Image generation** — having the assistant paint a picture of the destination city.
4. **Text-to-speech** — having the assistant speak its replies.
5. **`gr.Blocks`** — Gradio's third and most flexible UI mode, for when you need full
   control over layout and how components connect to callbacks.
6. Wiring all of the above together into one assistant that talks, shows images, and
   still uses tools/database lookups.

> **Note:** Each `gr.ChatInterface(...).launch()` / `ui.launch(...)` call opens a local
> web UI. Only the most recently *defined* `chat` function is used by a new launch, so
> run cells in order. Image generation costs a small amount per call — don't spam it.


## 1. Setup

Import what we need, load the API key, and initialize the client.

Your `.env` file (same folder as this notebook) should contain:

```
OPENAI_API_KEY=sk-...your-key...
```

This notebook assumes `prices.db` already exists (built in the previous notebook's
tool-calling exercise). If you're starting fresh, see the earlier notebook for how
the `prices` table is created and seeded.


In [ ]:
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please add OPENAI_API_KEY to your .env file")

MODEL = "gpt-4.1-mini"
openai_client = OpenAI()

DB = "prices.db"


In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""


## 2. Recap: the tool-calling assistant

This is the same pattern from the previous notebook: a `get_ticket_price` tool
backed by a SQLite database, described to the model via `price_function`, and a
`chat` function with a `while` loop that resolves tool calls until the model
returns a final answer.

We go straight to the final (while-loop, database-backed) version here, since
that's the one we'll build on for the rest of this notebook.


In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"


In [ ]:
get_ticket_price("Paris")


In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools


In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses


In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()


## 3. A bit more about what Gradio actually does

It's worth understanding the mechanics before customizing the UI further:

1. Gradio constructs a frontend **Svelte** app based on your Python description of
   the UI (the components you declare, like `gr.Textbox` or `gr.Chatbot`).
2. Gradio starts a server built on the **Starlette** web framework, listening on a
   free local port, which serves that Svelte app.
3. Gradio creates backend routes for your callbacks (like `chat()`), so the
   frontend can call your Python functions.

When Gradio generates the frontend, it wires up events (like clicking Submit, or
pressing Enter) to call the right backend route automatically. That's the whole
trick — simple in principle, but it produces something that feels close to magic.


## 4. Going multi-modal: image generation

We can use `gpt-image-1-mini` (the image model behind GPT-5) to generate a picture
representing a destination city. We'll wrap this in a function called `artist`.

**Price note:** each image generation costs a small amount (a few cents) — avoid
generating images in a loop or by accident.


In [ ]:
# Imports needed for handling generated images
import base64
from io import BytesIO
from PIL import Image
from IPython.display import display


In [ ]:
def artist(city):
    image_response = openai_client.images.generate(
        model="gpt-image-1-mini",
        prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
        size="1024x1024",
        n=1,
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))


In [ ]:
image = artist("New York City")
display(image)


## 5. Going multi-modal: text-to-speech

Similarly, we can have the assistant speak its replies out loud using OpenAI's
text-to-speech model. `talker` returns raw audio bytes, which Gradio's `gr.Audio`
component can play directly.

Try swapping `voice="onyx"` for `"alloy"` or `"coral"` to hear different voices.


In [ ]:
def talker(message):
    response = openai_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",    # Also try: "alloy" or "coral"
        input=message
    )
    return response.content


## 6. Bringing it all together

Now let's combine everything into one assistant:

1. **Multi-modal output** — text reply, spoken audio, and a generated image.
2. **Tool calling with database lookup** — same `get_ticket_price` pattern as before.
3. **A step toward an agentic workflow** — the assistant decides, per turn, whether
   it needs a tool, whether to speak, and whether to illustrate a city.

A few things change compared to the `gr.ChatInterface` version:

- `chat` now takes only `history` (no separate `message` argument), because we're
  moving to `gr.Blocks`, where *we* control exactly what gets passed in — more on
  this below.
- We track which cities were mentioned via tool calls (`handle_tool_calls_and_return_cities`),
  so we know what to illustrate.
- The function returns **three values**: the updated `history`, the spoken audio,
  and the generated image — matching the three output components we'll wire up
  in the UI next.


In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities


In [ ]:
def chat(history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])

    return history, voice, image


## 7. `gr.Blocks` — full control over the UI

Gradio has three ways to build a UI, each suited to a different level of control:

| Mode | Use case |
|---|---|
| `gr.Interface` | Standard, simple UIs — one function, some inputs, some outputs. |
| `gr.ChatInterface` | Standard chatbot UIs — history managed for you automatically. |
| `gr.Blocks` | Fully custom UIs — you place components yourself, and wire up exactly which events trigger which callbacks. |

We need `gr.Blocks` here because our layout is no longer "one chat box in, one
reply out" — we have a chatbot panel, an image panel, an audio player, and a
textbox, all laid out together, with two callbacks chained one after another.

### The event chain

```python
message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
    chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
)
```

This reads as: **when the textbox is submitted**,
1. First call `put_message_in_chatbot` — it clears the textbox and appends the
   user's message to the chat history immediately, so the UI feels responsive.
2. **`.then(...)`** — only after step 1 finishes, call `chat` with the *updated*
   chat history, and route its three return values into the chatbot display, the
   audio player, and the image panel respectively.

This two-step chain is a common Gradio pattern: show the user's message instantly,
then run the (slower) AI call as a follow-up step.


In [ ]:
# Callback that immediately reflects the user's message in the chat history,
# and clears the textbox
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]


In [ ]:
# UI definition
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    # Hooking up events to callbacks
    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))


> **Note on `auth=("ed", "bananas")`:** this adds simple username/password
> protection to the Gradio app — useful when sharing a link publicly. Change the
> credentials before using this for anything real, and never commit real
> credentials to a public repo (use environment variables instead).


## 8. Exercises and business applications

**Ideas to extend this further:**
- Add more tools — for example, one that actually books a flight (updating a
  `bookings` table, similar to how `set_ticket_price` updates `prices`).
- Apply this pattern to your own business: a customer support assistant, a new
  employee onboarding assistant, a product recommendation assistant — the same
  three ingredients (tools + database + multi-modal output) generalize well.
- Try adding authentication properly (e.g. reading credentials from environment
  variables instead of hardcoding them) if you plan to deploy this anywhere real.

This is a natural stopping point to reflect on what's been built across this
project: a conversational assistant that can look up real data, take actions via
tools, and communicate back through text, speech, and images — a genuine step
toward an **agentic** workflow, where the LLM is making decisions about what to
do next rather than just answering a single question.
